In [1]:
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt
%matplotlib inline

In [2]:
words = open('names.txt', 'r').read().splitlines()
words[:8]

['emma', 'olivia', 'ava', 'isabella', 'sophia', 'charlotte', 'mia', 'amelia']

In [3]:
len(words)

32033

In [4]:
# build the vocabulary of characters and mappings to/from numbers

chars = sorted(list(set(''.join(words))))
stoi = {s:i+1 for i,s in enumerate(chars)}
stoi['.'] = 0
itos = {i:s for s, i in stoi.items()}
print(itos)

{1: 'a', 2: 'b', 3: 'c', 4: 'd', 5: 'e', 6: 'f', 7: 'g', 8: 'h', 9: 'i', 10: 'j', 11: 'k', 12: 'l', 13: 'm', 14: 'n', 15: 'o', 16: 'p', 17: 'q', 18: 'r', 19: 's', 20: 't', 21: 'u', 22: 'v', 23: 'w', 24: 'x', 25: 'y', 26: 'z', 0: '.'}


In [5]:
# build the dataset

block_size = 3 # context length: how many characters do we take to predict the next one?
X, Y = [], []
for w in words[:5]:
  
  #print(w)
  context = [0] * block_size
  for ch in w + '.':
    ix = stoi[ch]
    X.append(context)
    Y.append(ix)
    print(''.join(itos[i] for i in context), '--->', itos[ix])
    context = context[1:] + [ix] # crop and append
  
X = torch.tensor(X)
Y = torch.tensor(Y)

... ---> e
..e ---> m
.em ---> m
emm ---> a
mma ---> .
... ---> o
..o ---> l
.ol ---> i
oli ---> v
liv ---> i
ivi ---> a
via ---> .
... ---> a
..a ---> v
.av ---> a
ava ---> .
... ---> i
..i ---> s
.is ---> a
isa ---> b
sab ---> e
abe ---> l
bel ---> l
ell ---> a
lla ---> .
... ---> s
..s ---> o
.so ---> p
sop ---> h
oph ---> i
phi ---> a
hia ---> .


In [6]:
X # 32 exxamples of 3 inputs to network that are integers

tensor([[ 0,  0,  0],
        [ 0,  0,  5],
        [ 0,  5, 13],
        [ 5, 13, 13],
        [13, 13,  1],
        [ 0,  0,  0],
        [ 0,  0, 15],
        [ 0, 15, 12],
        [15, 12,  9],
        [12,  9, 22],
        [ 9, 22,  9],
        [22,  9,  1],
        [ 0,  0,  0],
        [ 0,  0,  1],
        [ 0,  1, 22],
        [ 1, 22,  1],
        [ 0,  0,  0],
        [ 0,  0,  9],
        [ 0,  9, 19],
        [ 9, 19,  1],
        [19,  1,  2],
        [ 1,  2,  5],
        [ 2,  5, 12],
        [ 5, 12, 12],
        [12, 12,  1],
        [ 0,  0,  0],
        [ 0,  0, 19],
        [ 0, 19, 15],
        [19, 15, 16],
        [15, 16,  8],
        [16,  8,  9],
        [ 8,  9,  1]])

In [7]:
Y

tensor([ 5, 13, 13,  1,  0, 15, 12,  9, 22,  9,  1,  0,  1, 22,  1,  0,  9, 19,
         1,  2,  5, 12, 12,  1,  0, 19, 15, 16,  8,  9,  1,  0])

### First Layer

In [8]:
# first, build the embedding lookup table C
# if we look at the first piece "the embedding the integer" , we can interpret it as an integer indexing into lookup table C, 
# ..but equivalently we can think of this as First layer of bigger neural network. This layer has no non linearity and their weight matrix is C.

# whats there in embedding lookup tables ?
# index -> inputs -> integer ; columns -> dimensions -> embeddings

# we have 27 possible characters embedding into 2 dimensions
C = torch.randn((27, 2))
C

tensor([[-6.1174e-01, -6.5008e-01],
        [-1.5684e+00, -4.8896e-01],
        [-8.4340e-01, -5.4925e-01],
        [-3.5023e-01,  5.0616e-01],
        [-5.2351e-04,  7.9563e-01],
        [-1.2845e+00,  1.9410e+00],
        [-1.4605e+00,  2.9678e-01],
        [ 8.0563e-01,  1.2347e+00],
        [ 1.4729e+00,  1.5439e-01],
        [ 1.6992e-02,  4.2984e-01],
        [-8.6556e-01,  2.9714e-02],
        [-6.2397e-01,  1.4846e+00],
        [ 1.0762e+00, -5.1427e-01],
        [-1.5816e+00,  1.0623e-01],
        [ 4.5660e-01, -2.3838e+00],
        [-7.5413e-01, -2.1501e+00],
        [ 7.8878e-01,  1.3432e+00],
        [-3.5524e-01,  7.1029e-01],
        [ 1.9803e+00,  2.9754e-01],
        [ 9.5823e-02,  3.4201e-01],
        [-1.0696e+00, -1.4226e+00],
        [ 1.5378e+00,  2.0044e+00],
        [-4.4501e-01,  2.1908e-01],
        [-1.6940e+00,  4.3567e-01],
        [-4.6806e-01,  4.5286e-01],
        [-2.2330e-01, -1.0487e-01],
        [-1.3654e+00,  1.2222e+00]])

In [15]:
# so how we simultaneously embed 32 examples of 3 inputs in X
# pytorch allows to index with a list of numbers not just with a single value
# C[[5, 6, 7, 7]]

emb = C[X]
emb

tensor([[[-0.6117, -0.6501],
         [-0.6117, -0.6501],
         [-0.6117, -0.6501]],

        [[-0.6117, -0.6501],
         [-0.6117, -0.6501],
         [-1.2845,  1.9410]],

        [[-0.6117, -0.6501],
         [-1.2845,  1.9410],
         [-1.5816,  0.1062]],

        [[-1.2845,  1.9410],
         [-1.5816,  0.1062],
         [-1.5816,  0.1062]],

        [[-1.5816,  0.1062],
         [-1.5816,  0.1062],
         [-1.5684, -0.4890]],

        [[-0.6117, -0.6501],
         [-0.6117, -0.6501],
         [-0.6117, -0.6501]],

        [[-0.6117, -0.6501],
         [-0.6117, -0.6501],
         [-0.7541, -2.1501]],

        [[-0.6117, -0.6501],
         [-0.7541, -2.1501],
         [ 1.0762, -0.5143]],

        [[-0.7541, -2.1501],
         [ 1.0762, -0.5143],
         [ 0.0170,  0.4298]],

        [[ 1.0762, -0.5143],
         [ 0.0170,  0.4298],
         [-0.4450,  0.2191]],

        [[ 0.0170,  0.4298],
         [-0.4450,  0.2191],
         [ 0.0170,  0.4298]],

        [[-0.4450,  0

### Hidden Layer

In [10]:
# number of inputs for this is 6 , because 3 characters in context with 2 dimensions
# numer of neurons can be any number like say 100
W1 = torch.randn((6, 100))
b1 = torch.randn(100)

In [16]:
# to perform W1 * X, we need to convert X [32, ,3 ,2] -> [32, 6]. Only then [32, 6] * [6, 100].
# so we have to concat the 3 character embeddings

# first approach with hard coded values
# torch.cat([emb[:, 0, :], emb[:, 1, :], emb[:, 2, :]])

# pytorch way
# torch.cat(torch.unbind(emb, 1), 1)

# pytorch best way - taking advantage of strides/views in pytorch
embs = emb.view(32, 6)
embs

tensor([[-0.6117, -0.6501, -0.6117, -0.6501, -0.6117, -0.6501],
        [-0.6117, -0.6501, -0.6117, -0.6501, -1.2845,  1.9410],
        [-0.6117, -0.6501, -1.2845,  1.9410, -1.5816,  0.1062],
        [-1.2845,  1.9410, -1.5816,  0.1062, -1.5816,  0.1062],
        [-1.5816,  0.1062, -1.5816,  0.1062, -1.5684, -0.4890],
        [-0.6117, -0.6501, -0.6117, -0.6501, -0.6117, -0.6501],
        [-0.6117, -0.6501, -0.6117, -0.6501, -0.7541, -2.1501],
        [-0.6117, -0.6501, -0.7541, -2.1501,  1.0762, -0.5143],
        [-0.7541, -2.1501,  1.0762, -0.5143,  0.0170,  0.4298],
        [ 1.0762, -0.5143,  0.0170,  0.4298, -0.4450,  0.2191],
        [ 0.0170,  0.4298, -0.4450,  0.2191,  0.0170,  0.4298],
        [-0.4450,  0.2191,  0.0170,  0.4298, -1.5684, -0.4890],
        [-0.6117, -0.6501, -0.6117, -0.6501, -0.6117, -0.6501],
        [-0.6117, -0.6501, -0.6117, -0.6501, -1.5684, -0.4890],
        [-0.6117, -0.6501, -1.5684, -0.4890, -0.4450,  0.2191],
        [-1.5684, -0.4890, -0.4450,  0.2

In [ ]:
h = torch.tanh(embs @ W1 + b1) # theres broadcasting happening when adding bias

In [19]:
h.shape

torch.Size([32, 100])

### Final layer

In [ ]:
# input is 100 , and output is 27 because we have 27 possible characters

W2 = torch.randn((100, 27))
b2 = torch.randn(27)

In [ ]:
logits = h @ W2 + b2

In [ ]:
logits.shape

In [ ]:
counts = logits.exp() # using "fake counts" as interpretation only, its not actually counts

In [ ]:
prob = counts / counts.sum(1, keepdims=True)

In [ ]:
loss = -prob[torch.arange(32), Y].log().mean()
loss

In [20]:
# ------------ now made respectable :) ---------------